# Notebook 4 - Calculating the critical density if the system and comparing it with the mathematical predictions in 1 dimension
## 4.1) Calculating the early cascade condition
We will make use of this mathematical expression:

$\kappa = q_s + q_\ell$

Where $\kappa$ is the mean number of successful activation attempts made by one fired trap, $\q_s$ is the expected number of succesful short-range activations produced by one firing trap and $\q_l$ is the expected number of succesful long-range activations produced by one firing trap.

latex


This is a branching-process threshold calculation where the approach taken will be assuming that the active cells in the first three generations of the cascade remain active, then calculating number of expected activations per fired cell and converting it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def compute_kappa_mc(f_k, p_succ_long, p_succ_short, p_long, m=1,
                      n_samples=500_000, rng=None):
    """
    If the probabiloty distribution is unknown, then monte carlo approximations can be performed,
    should never be required as far as I am aware.
    """
    rng = rng or np.random.default_rng()

    q_s = (1 - p_long) * p_succ_short          # short branch: fixed success prob

    ks = np.array([f_k(rng) for _ in range(n_samples)])   # long branch: random k
    valid = ks >= 2                                        # k<2 attempts are wasted
    p_hits = np.where(valid, p_succ_long(np.clip(ks, 2, None)), 0.0)
    q_l = p_long * p_hits.mean()

    kappa = m * (q_s + q_l)
    return kappa, m * q_s, m * q_l

def rho_onset(f_k, p_succ_long, p_succ_short, p_long, m=1, rng=None):
    kappa, q_s, q_l = compute_kappa_mc(f_k, p_succ_long, p_succ_short, p_long, m, rng=rng)
    return 1.0 / kappa, kappa, q_s, q_l